# Train contextual token SAE (packed \(L^2\), 384-d context → 128-d recon)

This notebook mirrors `train_token_sae.py`: **MSE reconstruction** + **entropy penalty** on `p_softmax`. Splits data **50% train / 50% test** by protein file.

After training, the last cell saves **test-set** reconstructed pair tensors as **`{id}_reconstructed_pair.npy`** under **`{parent of Original_SAE}/reconstructed_pair_npy/`** (override with env **`RECONSTRUCTED_NPY_DIR`**). PDB generation is **not** run here—use `generate_pdbs_from_reconstructions.py` or `run_structure_module.py` later.

**Data layout:** `proteins_layer47/{protein_name}/{protein_name}_pair_block_47.npy` (example: `6tf4_A/6tf4_A_pair_block_47.npy`).  
The config cell picks **`Original_SAE/proteins_layer47`** if it exists, otherwise **`Autoencoder/proteins_layer47`** next to it (sibling of `Original_SAE`). Set **`PROTEIN_DATA_ROOT`** to override.  
Accepted filenames per protein folder:
- `{folder}_pair_block_47.npy` (same as the CLI script)
- `{folder}_pair_block47.npy` (no underscore before layer index)
- any `*pair_block*47*.npy` if the above are missing

**Memory:** Default is **`BATCH_SIZE=1`**, **`EVAL_BATCH_SIZE=1`**, **`DataParallel` off** (set env `TOKEN_SAE_DP=1` to enable). Each batch stacks **Σ L²** tokens — large proteins need small batches. Tune: `TOKEN_SAE_BATCH_SIZE`, `TOKEN_SAE_GRAD_ACCUM`, `TOKEN_SAE_EVAL_BATCH`. Optional: `export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to reduce fragmentation.

**If the GPU looks “full” but you closed the notebook:** another job or Python may still hold the device. On a cluster: `nvidia-smi` (see **PID** in **Processes**), then `kill <pid>`; for SLURM: `scancel <jobid>` / `squeue -u $USER`. `torch.cuda.empty_cache()` only releases PyTorch’s **unused** cache, not memory from live tensors.

**Multi-GPU:** Prefer `torchrun` + DDP for throughput; `DataParallel` is optional via `TOKEN_SAE_DP=1`.

In [ ]:
from __future__ import annotations

import gc
import os
import sys
import time
from pathlib import Path
from typing import List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Resolve directory containing sparse_autoencoder.py (works if cwd is repo root or elsewhere)
def _find_module_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "sparse_autoencoder.py").is_file():
            return p
        p = p.parent
    return Path.cwd().resolve()


NOTEBOOK_DIR = _find_module_dir()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np

from sparse_autoencoder import (
    ContextualPairDataset,
    ContextualTokenSAE,
    pack_context_collate,
    unpack_reconstructions,
)

In [ ]:
# --- Configuration ---
# Default data root (override with env PROTEIN_DATA_ROOT):
# 1) Original_SAE/proteins_layer47  if present
# 2) else Autoencoder/proteins_layer47 (sibling of Original_SAE — common layout)
# 3) else fall back to (1) path even if missing (clear error in next cell)
def _default_protein_root(nb_dir: Path) -> Path:
    local = nb_dir / "proteins_layer47"
    sibling = nb_dir.parent / "proteins_layer47"
    if local.is_dir():
        return local
    if sibling.is_dir():
        return sibling
    return local


PROTEIN_DATA_ROOT = Path(
    os.environ.get("PROTEIN_DATA_ROOT", str(_default_protein_root(NOTEBOOK_DIR)))
).resolve()
LAYER = 47

EPOCHS = 100
# Memory: collate stacks sum(L²) tokens — large proteins × many per batch OOMs fast.
BATCH_SIZE = int(os.environ.get("TOKEN_SAE_BATCH_SIZE", "1"))  # proteins per optimizer step (keep small)
GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("TOKEN_SAE_GRAD_ACCUM", "1"))  # >1: micro-batch; effective batch ≈ BATCH_SIZE × this
EVAL_BATCH_SIZE = int(os.environ.get("TOKEN_SAE_EVAL_BATCH", "1"))  # eval often fits 1 when train barely fits
LR = 1e-3
D_LATENT = 4096
TAU = 0.90
LAMBDA_ENTROPY = 0.01
SEED = 42

OUTPUT_DIR = Path(os.environ.get("TOKEN_SAE_OUTPUT", PROTEIN_DATA_ROOT / "token_sae_notebook_output")).resolve()

# Test reconstructions + PDBs: sibling of Original_SAE (parent of this notebook)
PARENT_DIR = NOTEBOOK_DIR.parent
RECONSTRUCTED_PDB_DIR = Path(
    os.environ.get("RECONSTRUCTED_PDB_DIR", str(PARENT_DIR / "reconstructed_pdbs"))
).resolve()
# Test-set reconstructed pair tensors only (.npy); PDB generation is separate
RECON_NPY_DIR = Path(
    os.environ.get("RECONSTRUCTED_NPY_DIR", str(PARENT_DIR / "reconstructed_pair_npy"))
).resolve()

# DataLoader: tune for H200 / fast storage
NUM_WORKERS = min(8, (os.cpu_count() or 4))
PERSISTENT_WORKERS = NUM_WORKERS > 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None

# CUDA — DataParallel duplicates inputs on GPU 0; leave off unless you need it (env TOKEN_SAE_DP=1)
USE_AMP = True  # bfloat16 on H200 is ideal when supported
USE_DATA_PARALLEL = os.environ.get("TOKEN_SAE_DP", "").lower() in ("1", "true", "yes")
USE_TORCH_COMPILE = False  # set True on PyTorch 2.x if compile works in your env

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # Faster matmul on Hopper / Ampere+ (does not affect fp64)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

In [ ]:
def discover_pair_paths(data_root: Path, layer: int = 47) -> List[str]:
    """
    Find pair-block npy files under data_root/<protein_name>/.
    Tries: {name}_pair_block_{layer}.npy, {name}_pair_block{layer}.npy, then glob *pair_block*47*.npy
    """
    paths: List[str] = []
    if not data_root.is_dir():
        raise FileNotFoundError(f"Data root does not exist or is not a directory: {data_root}")
    for subdir in sorted(data_root.iterdir()):
        if not subdir.is_dir():
            continue
        name = subdir.name
        candidates = [
            subdir / f"{name}_pair_block_{layer}.npy",
            subdir / f"{name}_pair_block{layer}.npy",
        ]
        chosen: Path | None = None
        for c in candidates:
            if c.is_file():
                chosen = c
                break
        if chosen is None:
            globs = list(subdir.glob("*pair_block*47*.npy")) + list(subdir.glob("*pair_block47*.npy"))
            if globs:
                chosen = sorted(globs, key=lambda p: p.name)[0]
        if chosen is not None:
            paths.append(str(chosen))
    return paths


pair_paths = discover_pair_paths(PROTEIN_DATA_ROOT, LAYER)
print(f"Found {len(pair_paths)} pair-block files under {PROTEIN_DATA_ROOT}")
for p in pair_paths[:5]:
    print("  ", p)
if len(pair_paths) > 5:
    print("  ...")
if not pair_paths:
    raise RuntimeError(
        f"No pair npy files found. Expected e.g. {PROTEIN_DATA_ROOT}/<protein>/<protein>_pair_block_47.npy"
    )

In [ ]:
# 50/50 train / test split at the protein (file) level
n_tot = len(pair_paths)
gen_split = torch.Generator().manual_seed(SEED)
perm = torch.randperm(n_tot, generator=gen_split)
n_test = n_tot // 2
test_paths = [pair_paths[i] for i in perm[:n_test].tolist()]
train_paths = [pair_paths[i] for i in perm[n_test:].tolist()]

train_dataset = ContextualPairDataset(train_paths, normalize=True)
test_dataset = ContextualPairDataset(test_paths, normalize=True)

pin = torch.cuda.is_available()
loader_kw = dict(
    collate_fn=pack_context_collate,
    num_workers=NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=PERSISTENT_WORKERS,
)
if PREFETCH_FACTOR is not None:
    loader_kw["prefetch_factor"] = PREFETCH_FACTOR

train_loader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE, **loader_kw)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=EVAL_BATCH_SIZE, **loader_kw)

print(
    f"Train: {len(train_paths)} | Test: {len(test_paths)} | "
    f"batch={BATCH_SIZE} accum={GRADIENT_ACCUMULATION_STEPS} (eff {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}) | "
    f"eval_batch={EVAL_BATCH_SIZE}"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(
        f"CUDA: {torch.cuda.get_device_name(0)} | devices: {torch.cuda.device_count()} | "
        f"bf16: {torch.cuda.is_bf16_supported()}"
    )

model = ContextualTokenSAE(
    d_context_in=384,
    d_latent=D_LATENT,
    d_recon_out=128,
    tau=TAU,
).to(device)

if USE_DATA_PARALLEL and device.type == "cuda" and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using nn.DataParallel across {torch.cuda.device_count()} GPUs")

if USE_TORCH_COMPILE and hasattr(torch, "compile"):
    model = torch.compile(model)
    print("torch.compile enabled")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

# bf16 on H200: no GradScaler; fp16 fallback uses GradScaler
USE_AUTOCAST = USE_AMP and device.type == "cuda"
use_bf16 = USE_AUTOCAST and torch.cuda.is_bf16_supported()
use_fp16 = USE_AUTOCAST and not use_bf16
scaler = torch.amp.GradScaler("cuda", enabled=use_fp16) if USE_AUTOCAST else None
if USE_AUTOCAST:
    print(f"AMP: {'bf16' if use_bf16 else 'fp16 + GradScaler'}")
else:
    print("AMP disabled (CPU or USE_AMP=False)")

if device.type == "cuda":
    gc.collect()
    torch.cuda.empty_cache()
    print(f"DataParallel: {USE_DATA_PARALLEL} (set TOKEN_SAE_DP=1 to enable multi-GPU)")

In [ ]:
def _forward_loss(
    packed_context: torch.Tensor,
    packed_targets: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    recon_packed, p_softmax, _ = model(packed_context)
    loss_recon = nn.functional.mse_loss(recon_packed, packed_targets)
    entropy = -(p_softmax * (p_softmax + 1e-10).log()).sum(dim=-1).mean()
    loss_total = loss_recon + LAMBDA_ENTROPY * entropy
    return loss_total, loss_recon, entropy


def train_one_epoch() -> Tuple[float, float, float]:
    model.train()
    total = total_r = total_h = 0.0
    n = 0
    acc = max(1, GRADIENT_ACCUMULATION_STEPS)
    pending = 0
    optimizer.zero_grad(set_to_none=True)

    for packed_context, packed_targets, _ in train_loader:
        packed_context = packed_context.to(device, non_blocking=pin)
        packed_targets = packed_targets.to(device, non_blocking=pin)

        if USE_AUTOCAST and use_bf16:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
            (loss_total / acc).backward()
        elif USE_AUTOCAST and use_fp16:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
            assert scaler is not None
            scaler.scale(loss_total / acc).backward()
        else:
            loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
            (loss_total / acc).backward()

        total += loss_total.item()
        total_r += loss_recon.item()
        total_h += entropy.item()
        n += 1
        pending += 1

        if pending == acc:
            if USE_AUTOCAST and use_fp16:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if USE_AUTOCAST and use_fp16:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            pending = 0

    if pending > 0:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.mul_(float(acc) / float(pending))
        if USE_AUTOCAST and use_fp16:
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        if USE_AUTOCAST and use_fp16:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    n = max(n, 1)
    return total / n, total_r / n, total_h / n


@torch.no_grad()
def evaluate(loader: DataLoader) -> Tuple[float, float, float]:
    model.eval()
    total = total_r = total_h = 0.0
    n = 0
    for packed_context, packed_targets, _ in loader:
        packed_context = packed_context.to(device, non_blocking=pin)
        packed_targets = packed_targets.to(device, non_blocking=pin)
        if USE_AUTOCAST and use_bf16:
            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
        elif USE_AUTOCAST and use_fp16:
            with torch.amp.autocast("cuda", dtype=torch.float16):
                loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
        else:
            loss_total, loss_recon, entropy = _forward_loss(packed_context, packed_targets)
        total += loss_total.item()
        total_r += loss_recon.item()
        total_h += entropy.item()
        n += 1
    n = max(n, 1)
    return total / n, total_r / n, total_h / n

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_val = float("inf")
t0 = time.perf_counter()

for epoch in range(EPOCHS):
    t_ep = time.perf_counter()
    tr_loss, tr_recon, tr_ent = train_one_epoch()
    te_loss, te_recon, te_ent = evaluate(test_loader)
    dt = time.perf_counter() - t_ep
    print(
        f"Epoch {epoch + 1}/{EPOCHS} ({dt:.1f}s) | "
        f"train {tr_loss:.6f} (recon {tr_recon:.6f}, H {tr_ent:.6f}) | "
        f"test {te_loss:.6f} (recon {te_recon:.6f}, H {te_ent:.6f})"
    )
    if device.type == "cuda":
        torch.cuda.empty_cache()
    if te_loss < best_val:
        best_val = te_loss
        to_save = model.module if isinstance(model, nn.DataParallel) else model
        torch.save(to_save.state_dict(), OUTPUT_DIR / "token_sae_best.pt")

to_save = model.module if isinstance(model, nn.DataParallel) else model
torch.save(to_save.state_dict(), OUTPUT_DIR / "token_sae_final.pt")
print(f"Done in {time.perf_counter() - t0:.1f}s. Checkpoints: {OUTPUT_DIR}")

In [ ]:
# Export test-set reconstructions as .npy only (run structure module / PDBs separately)
RECON_NPY_DIR.mkdir(parents=True, exist_ok=True)

if not test_paths:
    print("No test proteins in split; skip saving reconstructions.")
else:
    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()
    test_ds = ContextualPairDataset(test_paths, normalize=True)
    test_loader_single = DataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        collate_fn=pack_context_collate,
        num_workers=0,
        pin_memory=pin,
    )

    with torch.no_grad():
        for i, (packed_context, _tgt, original_shapes) in enumerate(test_loader_single):
            packed_context = packed_context.to(device, non_blocking=pin)
            if USE_AUTOCAST and use_bf16:
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    recon_packed, _, _ = m(packed_context)
            elif USE_AUTOCAST and use_fp16:
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    recon_packed, _, _ = m(packed_context)
            else:
                recon_packed, _, _ = m(packed_context)
            arr = unpack_reconstructions(recon_packed, original_shapes)[0].float().cpu().numpy()
            pid = Path(test_paths[i]).parent.name
            np.save(RECON_NPY_DIR / f"{pid}_reconstructed_pair.npy", arr)

    print(f"Saved {len(test_paths)} files to {RECON_NPY_DIR}  (e.g. {{id}}_reconstructed_pair.npy)")